## Aligning single-cell spatial transcriptomics datasets simulated with non-linear distortions (squidpy + moscot)

In this notebook, we simulate a warped ST dataset and evaluate how well moscot (via squidpy) can align it to the original.

The original notebook used STalign LDDMM. Here we use `moscot` optimal transport, which provides a different approach to spatial alignment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

### Load and warp data

In [ ]:
# Load MERFISH data
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)

xI0 = np.array(df1['center_x'])
yI0 = np.array(df1['center_y'])

fig, ax = plt.subplots()
ax.scatter(xI0, yI0, s=1, alpha=0.2, label='source init')
ax.legend(markerscale=10)

In [ ]:
# Warp the coordinates (simulating experimental distortion)
xI = pow(xI0, 1.25) / 10 + 500
yI = pow(yI0, 1.25) / 10 + 500

fig, ax = plt.subplots()
ax.scatter(xI, yI, s=1, alpha=0.2, label='source warped')
ax.legend(markerscale=10)

In [ ]:
# Target is the original unwarped coordinates
xJ = xI0
yJ = yI0

fig, ax = plt.subplots()
ax.scatter(xJ, yJ, s=1, alpha=0.2, c='#ff7f0e', label='target')
ax.legend(markerscale=10)

In [ ]:
# Plot overlay
fig, ax = plt.subplots()
ax.scatter(xI, yI, s=1, alpha=0.2, label='source warped')
ax.scatter(xJ, yJ, s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)

### Create AnnData objects

In [ ]:
# Source: warped coordinates
adata_source = ad.AnnData(X=np.zeros((len(xI), 1)))
adata_source.obsm['spatial'] = np.column_stack([xI, yI])

# Target: original coordinates
adata_target = ad.AnnData(X=np.zeros((len(xJ), 1)))
adata_target.obsm['spatial'] = np.column_stack([xJ, yJ])

### Perform alignment with moscot

In [ ]:
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

### Visualize results

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='source warped aligned')
ax.scatter(xJ, yJ, s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)

In [ ]:
plt.rcParams["figure.figsize"] = (12, 5)
fig, ax = plt.subplots(1, 2)
ax[0].scatter(xI, yI, s=0.5, alpha=0.1, label='source warped')
ax[0].scatter(xJ, yJ, s=0.5, alpha=0.1, label='target')
ax[1].scatter(aligned[:, 0], aligned[:, 1], s=0.5, alpha=0.1, label='source warped aligned')
ax[1].scatter(xJ, yJ, s=0.5, alpha=0.1, label='target')
ax[0].legend(markerscale=10, loc='lower left')
ax[1].legend(markerscale=10, loc='lower left')

### Evaluate performance

In [ ]:
from sklearn.metrics import mean_squared_error

err_init = mean_squared_error([xI0, yI0], [xI, yI], squared=False)
print(f"RMSE before alignment: {err_init:.2f}")

err_aligned = mean_squared_error([xI0, yI0], [aligned[:, 0], aligned[:, 1]], squared=False)
print(f"RMSE after alignment: {err_aligned:.2f}")